# Ultimate Guide to Web Scraping with Python Part 1: Requests and BeautifulSoup

In this tutorial, I will learn how to
- Request web pages
- Parse HTML
- Save and load scraped data
- Scrape multiple pages in a row.

## 1. Limit your impact when scraping
Every time you load a web page, you're making a request to a server.
With a Python script that can execute thousands of requests a second, 
if coded incorrectly, you could end up costing the website owner money and 
possibly bring down their site (DDoS attack).

**Every time we scrape, we should make only one request per page.**

_Hence why I shifted to Jupyter Notebook right here._

### Save HTML Locally

In [ ]:
# Function to save HTML web content locally
def save_html(html, path):
    with open(path, 'wb') as f: 
        # 'wb' means "write bytes" - avoids encoding issues
        f.write(html)

# save_html(r.content, 'google_com')
# Assuming r.content is HTML from google.com 
# We now have a "google_com" file that contains the HTML from google.com

NameError: name 'r' is not defined

In [ ]:
# Function to open/read HTML from local file
def open_html(path):
    with open(path, 'rb') as f:
        # 'rb' means "read bytes"
        return f.read()
    
html = open_html('google_com')
# Reads HTML from file "google_com"

# If our script fails / computer shuts down - we no longer
# need to request Google again, lessening the impact on their servers

**Advice**

Save every page you need and parse later when web scraping as a safety precaution.

### Scrapers and Bots

Each site usually has a robots.txt on the root of their domain, which explicitly states what bots are allowed to do on their site, for example:

In [ ]:
User-agent: * % name of the bot - * means these rules apply to all bots

% Rules that the bot must follow
Crawl-delay: 10 % number of seconds we wait before making another request
Allow: /pages/ % specific URLs we're allowed to request with bots e.g. allowed for example.com/pages/
Disallow: /scripts/  % not allowed to request e.g. not allowed for example.com/scripts/

%% Read rules in order 
Disallow: * % -> can't scrape anything...
Allow: /pages/ % -> except subfolder /pages/

**Note** : `robots.txt` works by exclusion. Anything **not explicitly disallowed is implicitly allowed**.

## 2. Scraping Project: Getting Media Bias Data

`save_html(r.content, 'google_com')`

With Python's `requests`library, we get a web page by using `get()` on the URL. The response is contained in `r`, which contains many things, but `r.content` gives us the HTML.

Once we have the HTML, we can save it to a file and parse it for the data we're interested in. 

In this project, we are scraping the _AllSides_ website, which has a media bias rating table.

In [3]:
import requests
import pprint

url = 'https://www.allsides.com/media-bias/media-bias-ratings'

r = requests.get(url)

# Print the first 100 chars to confirm we have the source of the page
pprint.pprint(r.content[:100])

(b'<!DOCTYPE html><html lang="en-US"><head><title>Just a moment...</title><meta'
 b' http-equiv="Content-Typ')


### Parsing HTML with BeautifulSoup
My soup is beautiful.

We have obtained the HTML from the _AllSides_ server, but now we need to parse the HTML with BeautifulSoup.

When we pass HTML to the BeautifulSoup constructor, this returns an object that we can then navigate like the original tree structure of the Document Object Model (DOM). _Side note: the DOM represents a web page as a tree-like structure of elements._

**insert image of a DOM here**

This way we can find elements using names of tags, classes, IDs, and through relationships to other elements, like getting the children and siblings of elements.

In [ ]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(r.content, 'html.parser')


<!DOCTYPE html>
<html lang="en-US"><head><title>Just a moment...</title><meta content="text/html; charset=utf-8" http-equiv="Content-Type"/><meta content="IE=Edge" http-equiv="X-UA-Compatible"/><meta content="noindex,nofollow" name="robots"/><meta content="width=device-width,initial-scale=1" name="viewport"/><style>*{box-sizing:border-box;margin:0;padding:0}html{line-height:1.15;-webkit-text-size-adjust:100%;color:#313131;font-family:system-ui,-apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,"Helvetica Neue",Arial,"Noto Sans",sans-serif,"Apple Color Emoji","Segoe UI Emoji","Segoe UI Symbol","Noto Color Emoji"}body{display:flex;flex-direction:column;height:100vh;min-height:100vh}.main-content{margin:8rem auto;padding-left:1.5rem;max-width:60rem}@media (width <= 720px){.main-content{margin-top:4rem}}.h2{line-height:2.25rem;font-size:1.5rem;font-weight:500}@media (width <= 720px){.h2{line-height:1.5rem;font-size:1.25rem}}#challenge-error-text{background-image:url("");background-repeat:n

### Making sense of the HTML - finding elements and data
To find the elements and data inside our HTML, we will use:
- `select_one` - this returns a single element.
- `select` - this returns a list of elements, even if only one item exists. 

Both methods use CSS selectors to find elements. 

### Ok, what the hell is a CSS selector?
_My thanks to GPT for helping explain this._

CCS stands for "Cascading Style Sheet". Cascading Style Sheets is what makes a website _look nice_! It controls aspects like:
- Font size
- Layout and spacing
- Backgrounds
- Positions of elements/pictures.

HTML builds the **structure**, CSS adds the **style**.

#### Nice. So what are tags? And why do I care?
Tags in CSS are usually **HTML tags**, also called **elements** - what you want to add style to. 

For example, in HTML, you might have:

In [ ]:
<p>This is a paragraph.</p>
<h1>This is a heading.</h1>
<button>Click me!</button>

SyntaxError: invalid syntax (938061686.py, line 1)

`<p>` is a tag/element for a paragraph.

In CSS, you "target" these tags to style them.

#### How CSS uses tags